#Tokenizer From Scratch


## Creating Token

###Download the dataset from the link https://github.com/Nuos/ml_LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/the-verdict.txt

In [ ]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text = f.read()
print("Total number of character:",len(raw_text))
raw_text[:99]

Total number of character: 20479


'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no '

In [ ]:
import re

text = "    This code, convert this sentence to tokens.   "
result = re.split(r'(\s)',text)
print(result)

['', ' ', '', ' ', '', ' ', '', ' ', 'This', ' ', 'code,', ' ', 'convert', ' ', 'this', ' ', 'sentence', ' ', 'to', ' ', 'tokens.', ' ', '', ' ', '', ' ', '']


In [ ]:
result = re.split(r'([,.]|\s)',text)
print(result)

['', ' ', '', ' ', '', ' ', '', ' ', 'This', ' ', 'code', ',', '', ' ', 'convert', ' ', 'this', ' ', 'sentence', ' ', 'to', ' ', 'tokens', '.', '', ' ', '', ' ', '', ' ', '']


In [ ]:
result = [ item for item in result if item.strip()]
print(result)

['This', 'code', ',', 'convert', 'this', 'sentence', 'to', 'tokens', '.']


In [ ]:
text.strip()

'This code, convert this sentence to tokens.'

In [ ]:
result = re.split(r'([,.:;/_!"()\']|--|\s)',text)
result = [item.strip() for item in result if item.strip()]
result

['This', 'code', ',', 'convert', 'this', 'sentence', 'to', 'tokens', '.']

In [ ]:
result = re.split(r'([,.:;/_!"()\']|--|\s)',raw_text)
result = [item.strip() for item in result if item.strip()]
print(result[:20])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


# Mapping the token to token_id by building a vocablary

In [ ]:
len(set(result))

1148

In [ ]:
all_word = sorted(set(result))
vocab_size = len(all_word)
vocab_size

1148

In [ ]:
vocab = { token:integer for integer,token in enumerate(all_word)}

In [ ]:
for item in vocab.items():
  print(item)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)
('How', 52)
('I', 53)
('If', 54)
('In', 55)
('It', 56)
('Jack', 57)
('Jove', 58)
('Just', 59)
('Lord', 60)
('Made', 61)
('Miss', 62)
('Money', 63)
('Monte', 64)
('Moon-dancers', 65)
('Mr', 66)
('Mrs', 67)
('My', 68)
('Never', 69)
('No', 70)
('Now', 71)
('Nutley', 72)
('Of', 73)
('Oh', 74)
('On', 75)
('Once', 76)
('Only', 77)
('

You can also say the process as a encoding as it is converting to token

In [ ]:
class SimpleTokenizerV1:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {i : s for s , i in vocab.items() }

  def encoder(self,text):
    preprocessed = re.split(r'([,.:;/_!"()\']|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids= [self.str_to_int[s] for s in preprocessed]
    return ids

  def decoder(self,ids):
    text = " ".join(self.int_to_str[i] for i in ids)
    "substitute the blank text to nothing before the expression markers "
    text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
    return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text = "I glanced after him, struck by his last word."
ids = tokenizer.encoder(text)
ids

[53, 497, 139, 551, 5, 958, 243, 554, 611, 1133, 7]

In [ ]:
text = tokenizer.decoder(ids)
text

'I glanced after him, struck by his last word.'

In [ ]:
text = "Hello what is python"
ids = tokenizer.encoder(text)
ids

KeyError: 'Hello'

For llm the same problem can arise if the vocab size is small


# Special Context Token

1. **<|endoftext|>** used to single the source of the text is ended and new source is starting . i.e it is inserted between two unrealted text
2. **<|unk|>**  is used for unknown text.

In [ ]:
all_token = sorted(list(set(result)))
all_token.extend(["<|endoftext|>","<|unk|>"])
vocab = { token : integer for integer , token in enumerate(all_token)}
len(vocab)

1150

In [ ]:
class SimpleTokenizerV2:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {i : s for s , i in vocab.items() }

  def encoder(self,text):
    preprocessed = re.split(r'([,.:;/_!"()\']|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [
        item if item in self.str_to_int
        else "<|unk|>" for item in preprocessed
    ]
    ids= [self.str_to_int[s] for s in preprocessed]
    return ids

  def decoder(self,ids):
    text = " ".join(self.int_to_str[i] for i in ids)
    "substitute the blank text to nothing before the expression markers "
    text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
    return text

In [ ]:
tokenizer2 = SimpleTokenizerV2(vocab)
text = "Hello what is python"
text1 = "Java is object orientend programming"
text = " <|endoftext|> ".join((text,text1))
ids = tokenizer2.encoder(text)
ids

[1149, 1106, 590, 1149, 1148, 1149, 590, 728, 1149, 1149]

In [ ]:
tokenizer2.decoder(ids)

'<|unk|> what is <|unk|> <|endoftext|> <|unk|> is object <|unk|> <|unk|>'

So far, we have discussed tokenization as an essential step in processing text as input to LLMs. Depending on the LLM, some researchers also consider additional special tokens such as the following:

[BOS] (beginning of sequence): This token marks the start of a text. It signifies to the LLM where a piece of content begins.

[EOS] (end of sequence): This token is positioned at the end of a text, and is especially useful when concatenating multiple unrelated texts, similar to <|endoftext|>. For instance, when combining two different Wikipedia articles or books, the [EOS] token indicates where one article ends and the next one begins.

[PAD] (padding): When training LLMs with batch sizes larger than one, the batch might contain texts of varying lengths. To ensure all texts have the same length, the shorter texts are extended or "padded" using the [PAD] token, up to the length of the longest text in the batch.

Note that the tokenizer used for GPT models does not need any of these tokens mentioned above but only uses an <|endoftext|> token for simplicity

the tokenizer used for GPT models also doesn't use an <|unk|> token for outof-vocabulary words. Instead, GPT models use a byte pair encoding tokenizer, which breaks down words into subword units


# Byte Pair Endcoding

In [ ]:
!pip3 install tiktoken

Types of tokenizer algorihtm

1. word based\
(what do you do with the out of vocabulary words, difference meaning of similar words boy , boys)
2. character based\
Problem
(very small vocabulary , meaning of the words is completely lost , the tokenized sequence is much logal than inital text)

3. sub word based\
Rule 1 :  do not split frequently used words into smaller subwords
Rule 2 : Split the rare word into smaller meaningfull subwords

Eg :- "boy" should not be split\
      "boys" should be split into "boy" and "s"


Byte Code Encoder

compression algroithm introudced in 1990s
it replace the most common token into the another varialbes
1. aaabdaaabac
2. QabdQabac
3. QYdQYac

In [ ]:
import importlib
import tiktoken

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
integers

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271,
 13]

In [ ]:
text = tokenizer.decode(integers)
text

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

# Creating Input-Target Pair

In [ ]:
with open("the-verdict.txt",'r',encoding="utf-8") as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [ ]:
enc_sample = enc_text[50:]

In [ ]:
enc_sample

[290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,
 286,
 526,
 383,
 1573,
 11,
 319,
 9074,
 13,
 536,
 5469,
 338,
 11914,
 11,
 33096,
 663,
 4808,
 3808,
 62,
 355,
 996,
 484,
 547,
 12548,
 287,
 281,
 13079,
 410,
 12523,
 286,
 22353,
 13,
 843,
 340,
 373,
 407,
 691,
 262,
 9074,
 13,
 536,
 48819,
 508,
 25722,
 276,
 13,
 11161,
 407,
 262,
 40123,
 18113,


In [ ]:
context_size = 10

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(x) #input
print("    ",y) # output

[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686]
     [4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976]


In [ ]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(context,"---->",desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257
[290, 4920, 2241, 287, 257] ----> 4489
[290, 4920, 2241, 287, 257, 4489] ----> 64
[290, 4920, 2241, 287, 257, 4489, 64] ----> 319
[290, 4920, 2241, 287, 257, 4489, 64, 319] ----> 262
[290, 4920, 2241, 287, 257, 4489, 64, 319, 262] ----> 34686
[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686] ----> 41976


In [ ]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context),"---->",tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a
 and established himself in a ---->  vill
 and established himself in a vill ----> a
 and established himself in a villa ---->  on
 and established himself in a villa on ---->  the
 and established himself in a villa on the ---->  Riv
 and established himself in a villa on the Riv ----> iera


### Efficent way to make input and target pair using pytorch tensor
later we are going to do batch processing we need to do it in efficient and structured way for the whole dataset

In [ ]:
from torch.utils.data import Dataset , DataLoader
import torch
class GptDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt,allowed_special={"<|endoftext|>"}) # Changed 'text' to 'txt' here

    for i in range(0 , len(token_ids)-max_length,stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]

      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self,idx):
    return self.input_ids[idx],self.target_ids[idx]

data loader steps

1. Initialize the tokenizer
2. Create dataset
3. drop table = True drpos the last batch if it is shorter than the specified batch_size to prevent loss spikes during training
4. the number of cpu processes to use for prepocessing


In [ ]:
def create_dataloader_v1(txt, batch_size = 4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
  #initalize the tokenizer

  tokenizer = tiktoken.get_encoding("gpt2")

  #Creating the dataset
  dataset = GptDatasetV1(txt,tokenizer,max_length,stride)
  #create data loader
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers
  )

  return dataloader


In [ ]:
with open("the-verdict.txt",'r',encoding="utf-8") as f:
  raw_text = f.read()

In [ ]:
import torch
print("pytorch version",torch.__version__)

dataloader = create_dataloader_v1(
    raw_text,batch_size=1,max_length=10,stride=1,shuffle=True
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

pytorch version 2.5.1+cu124


[tensor([[510,  26, 475, 314, 836, 470, 892, 286, 326,  11]]),
 tensor([[  26,  475,  314,  836,  470,  892,  286,  326,   11, 1770]])]

In [ ]:
second_batch = next(data_iter)
second_batch

[tensor([[10722,   292,  2280,   284,   616,  2583,   408,    25,   366,    40]]),
 tensor([[ 292, 2280,  284,  616, 2583,  408,   25,  366,   40, 1276]])]

Try to tweak the parameter to understand what affect which

# Token Embedding

In [ ]:
input_ids = torch.tensor([2,3,5,1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
embedding_layer.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

In [ ]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [ ]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


# Positional Embedding

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
with open("the-verdict.txt",'r',encoding="utf-8") as f:
  raw_text = f.read()
len(raw_text)

20479

In [ ]:
# Create the dataloader
dataloader3 = create_dataloader_v1(
    raw_text,batch_size=8,max_length=4,stride=4,shuffle=False
)

# Check the length of the dataset
dataset_length = len(dataloader3.dataset)


# Now try to iterate if there are expected batches
data_iter3 = iter(dataloader3)
first_batch3 = next(data_iter3)
print("Successfully retrieved the first batch.")
print(first_batch3)

Successfully retrieved the first batch.
[tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


2

In [ ]:
embedding_input = token_embedding_layer(first_batch3[0])
embedding_input.shape
#output dimention are batch_size X context size ( max_length ) X embedding dimention
# for now it is 8x4x256 a tensor of that dimention of input pair encoded throught bite pair encoder
output_embedding = token_embedding_layer(first_batch[1])

In [ ]:
# add postional embedding for the vector .
# our context size is the first dimention and to add this it should be equal to the ouput dimention
context_size = 4
pos_embedding_layer = torch.nn.Embedding(context_size,output_dim)
pos_embedding_layer.weight

Parameter containing:
tensor([[ 1.6459, -1.2114, -0.9353,  ..., -0.0461, -0.9937, -0.0690],
        [ 1.4213, -2.6221, -2.5158,  ..., -1.3071, -1.0946, -2.8479],
        [-0.0892,  0.5589,  0.5502,  ...,  0.0323, -0.4755, -0.0566],
        [-1.3638, -0.0531, -0.4435,  ...,  1.0636, -1.8907, -1.1526]],
       requires_grad=True)

In [ ]:
pos_embedding_layer.weight + embedding_input

tensor([[[ 2.1372, -0.0875,  0.5235,  ..., -0.4456, -2.8673, -0.2135],
         [ 1.8694, -2.3685, -2.7813,  ..., -0.8073, -2.2937, -4.0324],
         [-0.3399,  0.5042,  1.2190,  ...,  0.9941,  1.8982, -0.1095],
         [-0.4180,  0.8126,  1.1756,  ...,  0.6091, -2.6367, -0.8043]],

        [[ 3.1920,  0.5255, -1.7201,  ..., -0.1465, -0.1353, -0.4111],
         [-0.4409, -2.8135, -2.8970,  ..., -0.1851, -1.4442, -2.2389],
         [ 1.8955, -0.0894,  0.4087,  ..., -0.3518, -1.4110,  1.3912],
         [-0.3990,  1.2443, -2.0642,  ...,  2.2098, -0.3110, -0.7557]],

        [[ 0.8746, -0.5542, -0.7690,  ..., -0.8504, -0.9396,  0.6736],
         [ 2.2259, -2.1174, -1.2236,  ...,  0.1578, -0.6850, -2.5274],
         [-0.0098, -1.2047,  1.1252,  ...,  2.2146,  1.3476, -0.4201],
         [-0.9371, -0.1178,  0.1251,  ...,  0.5427, -0.5842, -0.3053]],

        ...,

        [[ 0.0304, -0.2504, -3.5790,  ..., -1.0105,  0.0951,  1.5693],
         [ 1.0228, -3.5456, -3.8321,  ..., -2.4652, -2.22